In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/09_dim_observation_type
# Purpose  : SCD Type 1 for observation types (213 distinct codes based on profiling).
#            Decouples code, description, and units from the 16.2M row fact table.
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.view(name="v_observation_type_changes")
def v_observation_type_changes():
    df = dlt.read_stream("covid19_lakehouse.silver.observations")
    return df.select(
        F.xxhash64("observation_code").alias("observation_type_key"),
        "observation_code",
        "observation_description",
        "units",
        "_ingested_at",
    )


dlt.create_streaming_table("covid19_lakehouse.gold.dim_observation_type")

dlt.apply_changes(
    target="covid19_lakehouse.gold.dim_observation_type",
    source="v_observation_type_changes",
    keys=["observation_code"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=1,
)